In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np
import pandas as pd

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

data/metadata.csv
data/test_Q.csv
data/train_QA.csv


In [14]:
# ============================================
# WATTBOT RAG - COMPLETE WORKING KAGGLE VERSION
# ============================================

import subprocess
import sys

print("Installing required packages with compatible versions...")
packages = [
    'sentence-transformers>=2.7.0',
    'faiss-cpu',
    'PyPDF2',
    'transformers>=4.36.0',
    'torch>=2.0.1',
    'huggingface_hub>=0.16.4'
]

for package in packages:
    print(f"Installing {package}...")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
    except Exception as e:
        print(f"Warning: Could not install {package}: {e}")

print("✅ Package installation completed!")

Installing required packages with compatible versions...
Installing sentence-transformers>=2.7.0...
Installing faiss-cpu...
Installing PyPDF2...
Installing transformers>=4.36.0...
Installing torch>=2.0.1...
Installing huggingface_hub>=0.16.4...
✅ Package installation completed!


In [15]:
import pandas as pd
import numpy as np
import os
import re
import json
from typing import List, Dict, Tuple, Optional, Any
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

try:
    import PyPDF2
except ImportError:
    print("Warning: PyPDF2 not available")

try:
    from sentence_transformers import SentenceTransformer
    import torch
    from transformers import AutoTokenizer, pipeline
    import faiss
    print("✅ All ML libraries loaded successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please run the package installation cell first")

from tqdm.auto import tqdm
import hashlib
import pickle
from pathlib import Path
import gc
from datetime import datetime

def read_csv_with_encoding_detection(file_path):
    """Read CSV with automatic encoding detection"""
    encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'utf-16']
    
    for encoding in encodings_to_try:
        try:
            print(f"Trying to read {file_path} with {encoding} encoding...")
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✅ Successfully read with {encoding} encoding")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Error with {encoding}: {e}")
            continue
    
    try:
        print(f"Trying to read {file_path} with UTF-8 and errors='ignore'...")
        df = pd.read_csv(file_path, encoding='utf-8', errors='ignore')
        print(f"✅ Read with UTF-8 (ignoring errors)")
        return df
    except Exception as e:
        print(f"❌ Failed to read {file_path}: {e}")
        raise

✅ All ML libraries loaded successfully


In [ ]:
# ----------------------------
# 1️⃣ read JSON config
# ----------------------------
import json

with open("config.json", "r") as f:
    config = json.load(f)

for directory in [config['cache_dir'], config['text_cache_dir']]:
    os.makedirs(directory, exist_ok=True)

In [17]:
# ----------------------------
# 2	Document Processor
# ----------------------------
class ImprovedDocumentProcessor:
    def __init__(self, cache_dir, text_cache_dir):
        self.cache_dir = cache_dir
        self.text_cache_dir = text_cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        os.makedirs(text_cache_dir, exist_ok=True)
        
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (compatible; AcademicCrawler/1.0)'
        })
    
    def get_text_cache_path(self, doc_id: str) -> str:
        return os.path.join(self.text_cache_dir, f"{doc_id}.txt")
    
    def load_cached_text(self, doc_id: str) -> Optional[str]:
        cache_path = self.get_text_cache_path(doc_id)
        if os.path.exists(cache_path):
            try:
                with open(cache_path, 'r', encoding='utf-8') as f:
                    return f.read()
            except Exception as e:
                print(f"Cache read error for {doc_id}: {e}")
        return None
    
    def save_text_cache(self, doc_id: str, text: str):
        cache_path = self.get_text_cache_path(doc_id)
        try:
            with open(cache_path, 'w', encoding='utf-8') as f:
                f.write(text)
        except Exception as e:
            print(f"Cache write error for {doc_id}: {e}")
    
    def download_pdf(self, url: str, doc_id: str) -> Optional[str]:
        if config['skip_download']:
            return None
            
        pdf_path = os.path.join(self.cache_dir, f"{doc_id}.pdf")
        if os.path.exists(pdf_path):
            return pdf_path
        
        urls_to_try = [url]
        if 'arxiv.org' in url:
            if '/abs/' in url:
                pdf_url = url.replace('/abs/', '/pdf/') + '.pdf'
                urls_to_try.append(pdf_url)
            elif '/pdf/' not in url:
                urls_to_try.append(url + '.pdf')
        
        for attempt_url in urls_to_try:
            try:
                print(f"Downloading: {attempt_url[:60]}...")
                response = self.session.get(attempt_url, timeout=30)
                if response.status_code == 200 and len(response.content) > 1000:
                    with open(pdf_path, 'wb') as f:
                        f.write(response.content)
                    print(f"✅ Downloaded {doc_id}")
                    return pdf_path
                else:
                    print(f"❌ Failed {doc_id}: Status {response.status_code}")
            except Exception as e:
                print(f"❌ Download error for {doc_id}: {str(e)[:50]}")
                continue
        
        return None
    
    def extract_text_from_pdf(self, pdf_path: str) -> str:
        text = ""
        if not os.path.exists(pdf_path):
            return text
        try:
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                num_pages = min(len(pdf_reader.pages), config['max_pdf_pages'])
                for i in range(num_pages):
                    try:
                        page = pdf_reader.pages[i]
                        page_text = page.extract_text()
                        if page_text and len(page_text.strip()) > 10:
                            text += page_text + "\n"
                    except Exception as e:
                        print(f"Page {i} extraction error: {e}")
                        continue
        except Exception as e:
            print(f"PDF extraction error: {str(e)[:100]}")
        return text.strip()
    
    def clean_text(self, text: str) -> str:
        if not text:
            return ""
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)\[\]\"\'\/\%\$\&\@\#\+\=]', ' ', text)
        text = re.sub(r'(\w)-\s+(\w)', r'\1\2', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

In [18]:
# ----------------------------
# 3️⃣ Text Chunker
# ----------------------------
class SmartTextChunker:
    def __init__(self, chunk_size=300, overlap=50):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def chunk_text(self, text: str, doc_id: str) -> List[Dict]:
        if not text or len(text) < config['min_chunk_length']:
            return []
        text = re.sub(r'\s+', ' ', text)
        text = text[:config['max_text_length']]
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks = []
        current_chunk = []
        current_length = 0
        chunk_id = 0
        
        for sentence in sentences:
            if not sentence.strip():
                continue
            sentence_length = len(sentence.split())
            if current_length + sentence_length > self.chunk_size and current_chunk:
                chunk_text = ' '.join(current_chunk)
                if len(chunk_text) >= config['min_chunk_length']:
                    chunks.append({
                        'doc_id': doc_id,
                        'chunk_id': f"{doc_id}_chunk_{chunk_id}",
                        'text': chunk_text,
                        'chunk_num': chunk_id
                    })
                    chunk_id += 1
                # Handle overlap
                overlap_sentences = []
                overlap_length = 0
                for sent in reversed(current_chunk):
                    sent_len = len(sent.split())
                    if overlap_length + sent_len <= self.overlap:
                        overlap_sentences.insert(0, sent)
                        overlap_length += sent_len
                    else:
                        break
                current_chunk = overlap_sentences
                current_length = overlap_length
            current_chunk.append(sentence)
            current_length += sentence_length
        
        if current_chunk:
            chunk_text = ' '.join(current_chunk)
            if len(chunk_text) >= config['min_chunk_length']:
                chunks.append({
                    'doc_id': doc_id,
                    'chunk_id': f"{doc_id}_chunk_{chunk_id}",
                    'text': chunk_text,
                    'chunk_num': chunk_id
                })
        return chunks

In [19]:
# ----------------------------
# 4️⃣ RAG System
# ----------------------------
class EnhancedRAG:
    def __init__(self, config):
        self.config = config
        self.doc_processor = ImprovedDocumentProcessor(
            config['cache_dir'], config['text_cache_dir']
        )
        self.chunker = SmartTextChunker(config['chunk_size'], config['chunk_overlap'])
        
        print("Initializing models...")
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {device}")
        try:
            self.embedding_model = SentenceTransformer(config['embedding_model'], device=device)
            print(f"✅ Loaded embedding model: {config['embedding_model']}")
            self.qa_pipeline = pipeline(
                "question-answering",
                model=config['qa_model'],
                device=0 if device == 'cuda' else -1
            )
            print(f"✅ Loaded QA model: {config['qa_model']}")
        except Exception as e:
            print(f"❌ Model loading error: {e}")
            raise
        
        self.chunks = []
        self.embeddings = None
        self.metadata_df = None
        self.index = None
        self.doc_id_to_metadata = {}
    
    def process_documents(self, metadata_df: pd.DataFrame):
        self.metadata_df = metadata_df
        
        for _, row in metadata_df.iterrows():
            self.doc_id_to_metadata[row['id']] = {
                'title': row['title'],
                'year': row.get('year', ''),
                'citation': row.get('citation', ''),
                'url': row['url']
            }
        
        if os.path.exists(config['chunks_cache']) and not config['use_sample_mode']:
            print("Loading cached chunks...")
            try:
                with open(config['chunks_cache'], 'rb') as f:
                    self.chunks = pickle.load(f)
                print(f"✅ Loaded {len(self.chunks)} cached chunks")
                return
            except Exception as e:
                print(f"Cache loading failed: {e}, reprocessing...")
        
        print("Processing documents...")
        all_chunks = []
        
        if config['use_sample_mode']:
            metadata_df = metadata_df.head(10)
            print("📝 SAMPLE MODE: Processing only 10 documents")
        elif config['max_docs_to_process']:
            metadata_df = metadata_df.head(config['max_docs_to_process'])
        
        success_count = 0
        fail_count = 0
        
        for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Processing docs"):
            doc_id = str(row['id'])
            url = row['url']
            
            text = self.doc_processor.load_cached_text(doc_id)
            
            if not text:
                if pd.isna(url) or not str(url).startswith('http'):
                    print(f"❌ Invalid URL for {doc_id}")
                    fail_count += 1
                    continue
                
                try:
                    pdf_path = self.doc_processor.download_pdf(str(url), doc_id)
                    if pdf_path:
                        text = self.doc_processor.extract_text_from_pdf(pdf_path)
                        text = self.doc_processor.clean_text(text)
                        
                        if text and len(text) > 100:
                            self.doc_processor.save_text_cache(doc_id, text)
                            print(f"✅ Processed {doc_id}: {len(text)} chars")
                        else:
                            print(f"❌ Empty text for {doc_id}")
                    else:
                        print(f"❌ Download failed for {doc_id}")
                except Exception as e:
                    print(f"❌ Error with {doc_id}: {str(e)[:50]}")
                    fail_count += 1
                    continue
            else:
                print(f"✅ Using cached text for {doc_id}")
            
            if text and len(text) > 100:
                chunks = self.chunker.chunk_text(text, doc_id)
                
                for chunk in chunks:
                    chunk.update({
                        'title': str(row['title'])[:200] if 'title' in row else '',
                        'year': str(row.get('year', '')),
                        'url': str(url)
                    })
                
                all_chunks.extend(chunks)
                success_count += 1
                print(f"✅ Created {len(chunks)} chunks for {doc_id}")
            else:
                fail_count += 1
                print(f"❌ No valid text for {doc_id}")
            
            if (idx + 1) % 5 == 0:
                gc.collect()
        
        self.chunks = all_chunks
        
        print(f"\n📊 Processing Summary:")
        print(f"  ✅ Success: {success_count} documents")
        print(f"  ❌ Failed: {fail_count} documents")  
        print(f"  📄 Total chunks: {len(self.chunks)}")
        
        if not config['use_sample_mode'] and self.chunks:
            try:
                with open(config['chunks_cache'], 'wb') as f:
                    pickle.dump(self.chunks, f)
                print("✅ Chunks cached successfully")
            except Exception as e:
                print(f"❌ Caching failed: {e}")
    
    def create_embeddings(self):
        if os.path.exists(config['embeddings_cache']) and not config['use_sample_mode']:
            print("Loading cached embeddings...")
            try:
                with open(config['embeddings_cache'], 'rb') as f:
                    self.embeddings = pickle.load(f)
                print(f"✅ Loaded cached embeddings: {self.embeddings.shape}")
                self._build_index()
                return
            except Exception as e:
                print(f"Embedding cache loading failed: {e}")
        
        if not self.chunks:
            print("❌ No chunks to embed!")
            return
        
        print(f"🧮 Creating embeddings for {len(self.chunks)} chunks...")
        texts = [chunk['text'][:1000] for chunk in self.chunks]
        
        try:
            all_embeddings = []
            batch_size = config['batch_size']
            
            for i in tqdm(range(0, len(texts), batch_size), desc="Creating embeddings"):
                batch = texts[i:i+batch_size]
                batch_embeddings = self.embedding_model.encode(
                    batch,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=True
                )
                all_embeddings.append(batch_embeddings)
                
                if (i // batch_size + 1) % 5 == 0:
                    gc.collect()
            
            self.embeddings = np.vstack(all_embeddings)
            print(f"✅ Created embeddings: {self.embeddings.shape}")
            
            if not config['use_sample_mode']:
                try:
                    with open(config['embeddings_cache'], 'wb') as f:
                        pickle.dump(self.embeddings, f)
                    print("✅ Embeddings cached")
                except Exception as e:
                    print(f"❌ Embedding caching failed: {e}")
            
            self._build_index()
            
        except Exception as e:
            print(f"❌ Embedding creation failed: {e}")
            raise
    
    def _build_index(self):
        if self.embeddings is None:
            print("❌ No embeddings available for indexing")
            return
        
        try:
            print("🔍 Building search index...")
            dimension = self.embeddings.shape[1]
            
            self.index = faiss.IndexFlatIP(dimension)
            self.index.add(self.embeddings.astype('float32'))
            print(f"✅ Index built: {self.index.ntotal} vectors")
            
        except Exception as e:
            print(f"❌ Index building failed: {e}")
            raise
    
    def retrieve_chunks(self, query: str, k: int = 7) -> List[Dict]:
        if not self.index or self.index.ntotal == 0:
            return []
        
        try:
            query_embedding = self.embedding_model.encode(
                [query],
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=True
            )
            
            k = min(k, self.index.ntotal)
            scores, indices = self.index.search(
                query_embedding.astype('float32'), k
            )
            
            retrieved_chunks = []
            for i, idx in enumerate(indices[0]):
                if idx < len(self.chunks):
                    chunk = self.chunks[idx].copy()
                    chunk['score'] = float(scores[0][i])
                    retrieved_chunks.append(chunk)
            
            retrieved_chunks.sort(key=lambda x: x['score'], reverse=True)
            return retrieved_chunks
            
        except Exception as e:
            print(f"❌ Retrieval error: {str(e)[:100]}")
            return []
    
    def extract_answer_advanced(self, question: str, chunks: List[Dict]) -> Dict:
        # CORRECTED: Competition requires "is_blank" for unanswerable questions
        default_response = {
            'answer': "Unable to answer with confidence based on the provided documents.",
            'answer_value': "is_blank",
            'answer_unit': "is_blank",
            'ref_id': "is_blank",
            'ref_url': "is_blank",
            'supporting_materials': "is_blank",
            'explanation': "is_blank"
        }
        
        if not chunks:
            return default_response
        
        context_parts = []
        ref_ids = set()
        ref_urls = set()
        
        for chunk in chunks[:5]:
            if chunk.get('score', 0) > config['similarity_threshold']:
                context_parts.append(chunk['text'])
                ref_ids.add(chunk['doc_id'])
                ref_urls.add(chunk.get('url', ''))
        
        if not context_parts:
            return default_response
        
        combined_context = " ".join(context_parts)[:config['max_context_length']]
        
        try:
            qa_result = self.qa_pipeline(
                question=question,
                context=combined_context,
                max_answer_len=100
            )
            
            if not qa_result or qa_result.get('score', 0) < 0.01:
                return default_response
            
            answer_text = qa_result['answer']
            
            answer_value, answer_unit = self._parse_answer_improved(answer_text)
            
            start = max(0, qa_result.get('start', 0) - 50)
            end = min(len(combined_context), qa_result.get('end', 50) + 50)
            supporting = combined_context[start:end].strip()
            
            supporting = re.sub(r'\s+', ' ', supporting)
            if len(supporting) > 200:
                supporting = supporting[:200] + "..."
            
            return {
                'answer': answer_text,
                'answer_value': answer_value,
                'answer_unit': answer_unit,
                'ref_id': ', '.join(sorted(ref_ids)),
                'ref_url': ', '.join([u for u in sorted(ref_urls) if u and u != 'nan']),
                'supporting_materials': f'"{supporting}"',
                'explanation': f"Confidence: {qa_result['score']:.3f}"
            }
            
        except Exception as e:
            print(f"❌ Answer extraction error: {str(e)[:50]}")
            return default_response
    
    def _parse_answer_improved(self, answer: str) -> Tuple[str, str]:
        if not answer:
            return "is_blank", "is_blank"
        
        answer = answer.strip()
        
        if answer.upper() in ['TRUE', 'FALSE', 'YES', 'NO']:
            value = '1' if answer.upper() in ['TRUE', 'YES'] else '0'
            return value, 'boolean'
        
        patterns = [
            (r'([\d,]+\.?\d*)\s*([a-zA-Z]+(?:\s+[a-zA-Z]+)*)', 'num_unit'),
            (r'([\d,]+\.?\d*)\s*(%)', 'percentage'),
            (r'\$\s*([\d,]+\.?\d*)', 'currency'),
            (r'([\d,]+\.?\d*)', 'number'),
        ]
        
        for pattern, pattern_type in patterns:
            match = re.search(pattern, answer)
            if match:
                if pattern_type == 'num_unit':
                    value = match.group(1).replace(',', '')
                    unit = match.group(2).strip()
                    return value, unit
                elif pattern_type == 'percentage':
                    value = match.group(1).replace(',', '')
                    return value, 'percent'
                elif pattern_type == 'currency':
                    value = match.group(1).replace(',', '')
                    return value, 'USD'
                else:
                    value = match.group(1).replace(',', '')
                    return value, "is_blank"
        
        return answer, "is_blank"
    
    def answer_question(self, question: str) -> Dict:
        if not self.chunks or not self.index:
            return {
                'answer': "System not ready: No processed documents available.",
                'answer_value': "is_blank",
                'answer_unit': "is_blank",
                'ref_id': "is_blank",
                'ref_url': "is_blank",
                'supporting_materials': "is_blank",
                'explanation': "is_blank"
            }
        
        chunks = self.retrieve_chunks(question, k=config['max_chunks_per_query'])
        return self.extract_answer_advanced(question, chunks)

In [ ]:
def main():
    print("="*60)
    print(" WATTBOT RAG SYSTEM - COMPLETE WORKING VERSION")
    print("="*60)
    
    start_time = datetime.now()
    
    print(f"\n🏗️ Environment Check:")
    print(f"  Working directory: {os.getcwd()}")
    print(f"  Expected input path: {config['base_path']}")
    print(f"  Output path: {config['submission_path']}")
    
    possible_locations = [
        'data'
    ]
    
    found_location = None
    for location in possible_locations:
        if os.path.exists(location):
            files = os.listdir(location)
            print(f"\n📂 Found directory: {location}")
            print(f"   Files: {files}")
            
            if 'metadata.csv' in files and 'test_Q.csv' in files:
                found_location = location
                print(f"✅ Using files from: {location}")
                config['base_path'] = location
                config['metadata_path'] = os.path.join(location, 'metadata.csv')
                config['test_q_path'] = os.path.join(location, 'test_Q.csv')
                config['train_qa_path'] = os.path.join(location, 'train_QA.csv')
                break
    
    if not found_location:
        print("❌ Could not find required CSV files!")
        return None
    
    print("\n📁 Loading data files with encoding detection...")
    try:
        metadata_df = read_csv_with_encoding_detection(config['metadata_path'])
        test_q_df = read_csv_with_encoding_detection(config['test_q_path'])
        
        train_qa_df = None
        if os.path.exists(config['train_qa_path']):
            try:
                train_qa_df = read_csv_with_encoding_detection(config['train_qa_path'])
            except Exception as e:
                print(f"Warning: Could not load training data: {e}")
        
        print(f"  📄 Documents: {len(metadata_df)}")
        print(f"  🎯 Test questions: {len(test_q_df)}")
        if train_qa_df is not None:
            print(f"  ❓ Training questions: {len(train_qa_df)}")
        
        print(f"\n📋 Sample metadata columns: {list(metadata_df.columns)}")
        print(f"📋 Sample test question columns: {list(test_q_df.columns)}")
        
    except Exception as e:
        print(f"❌ Error loading data files: {e}")
        return None
    
    if config['use_sample_mode']:
        print("\n⚠️  SAMPLE MODE - Processing limited documents")
        print("    Set config['use_sample_mode'] = False for full processing")
    
    try:
        print("\n🚀 Initializing RAG system...")
        rag = EnhancedRAG(config)
        
        print("\n📚 Processing documents...")
        rag.process_documents(metadata_df)
        
        if not rag.chunks:
            print("❌ No chunks created! Check document processing.")
            return None
        
        print("\n🧮 Creating embeddings...")
        rag.create_embeddings()
        
        if rag.embeddings is None:
            print("❌ No embeddings created! Check embedding process.")
            return None
        
    except Exception as e:
        print(f"❌ System initialization failed: {e}")
        return None
    
    if train_qa_df is not None and len(train_qa_df) > 0 and config['use_sample_mode']:
        print("\n🧪 Testing on sample training questions...")
        test_samples = min(3, len(train_qa_df))
        
        for i in range(test_samples):
            row = train_qa_df.iloc[i]
            print(f"\n  Q{i+1}: {row['question'][:80]}...")
            result = rag.answer_question(row['question'])
            print(f"  Generated: {result['answer'][:80]}")
            print(f"  Expected: {str(row['answer'])[:80]}")
            print(f"  Confidence: {result['explanation']}")
    
    print(f"\n🎯 Generating predictions for {len(test_q_df)} questions...")
    predictions = []
    
    try:
        for idx, row in tqdm(test_q_df.iterrows(), total=len(test_q_df), desc="Answering"):
            result = rag.answer_question(row['question'])
            
            # Ensure all fields are strings and handle any None values
            prediction = {
                'id': str(row['id']) if pd.notna(row['id']) else "",
                'question': str(row['question']) if pd.notna(row['question']) else "",
                'answer': str(result['answer']) if pd.notna(result['answer']) else "Unable to answer with confidence based on the provided documents.",
                'answer_value': str(result['answer_value']) if pd.notna(result['answer_value']) else "is_blank",
                'answer_unit': str(result['answer_unit']) if pd.notna(result['answer_unit']) else "is_blank",
                'ref_id': str(result['ref_id']) if pd.notna(result['ref_id']) else "is_blank",
                'ref_url': str(result['ref_url']) if pd.notna(result['ref_url']) else "is_blank",
                'supporting_materials': str(result['supporting_materials']) if pd.notna(result['supporting_materials']) else "is_blank",
                'explanation': str(result['explanation']) if pd.notna(result['explanation']) else "is_blank"
            }
            predictions.append(prediction)
            
            if (idx + 1) % 50 == 0:
                gc.collect()
        
        print(f"\n💾 Creating submission file...")
        submission_df = pd.DataFrame(predictions)
        
        # COMPREHENSIVE NULL CLEANING WITH PROPER DEFAULTS
        print("🧹 Comprehensive null value cleaning...")
        
        # Define proper defaults for each column per competition requirements
        column_defaults = {
            'id': '',
            'question': '',
            'answer': 'Unable to answer with confidence based on the provided documents.',
            'answer_value': 'is_blank',  # Competition requirement
            'answer_unit': 'is_blank',   # Competition requirement
            'ref_id': 'is_blank',        # Competition requirement
            'ref_url': 'is_blank',       # Competition requirement  
            'supporting_materials': 'is_blank',  # Competition requirement
            'explanation': 'is_blank'    # Competition requirement
        }
        
        # Fill nulls with proper defaults
        for col, default_value in column_defaults.items():
            if col in submission_df.columns:
                # First handle pandas nulls
                submission_df[col] = submission_df[col].fillna(default_value)
                # Convert to string and replace string representations of None
                submission_df[col] = submission_df[col].astype(str)
                submission_df[col] = submission_df[col].replace(['None', 'nan', 'NaN', 'null', 'NULL'], default_value)
        
        # Final safety check
        submission_df = submission_df.fillna('')
        
        # Verify no nulls remain
        null_count = submission_df.isnull().sum().sum()
        print(f"Null values after comprehensive cleaning: {null_count}")
        
        # Final validation
        print("\n🔍 Final validation check:")
        for col in submission_df.columns:
            unique_vals = submission_df[col].unique()
            problematic = [v for v in unique_vals if pd.isna(v) or (isinstance(v, str) and v.lower() in ['none', 'nan', 'null'])]
            if problematic:
                print(f"  ⚠️ Column {col} has problematic values: {problematic}")
                # Fix any remaining issues
                submission_df[col] = submission_df[col].replace(problematic, column_defaults.get(col, ''))
            else:
                print(f"  ✅ Column {col} looks clean")
        
        os.makedirs(os.path.dirname(config['submission_path']), exist_ok=True)
        submission_df.to_csv(config['submission_path'], index=False)
        
        elapsed = datetime.now() - start_time
        print("\n" + "="*60)
        print(" KAGGLE PROCESSING COMPLETE!")
        print("="*60)
        print(f"⏱️  Total time: {elapsed}")
        print(f"📊 Documents processed: {len([c for c in set(chunk['doc_id'] for chunk in rag.chunks)])}")
        print(f"📄 Text chunks created: {len(rag.chunks)}")
        print(f"✅ Questions answered: {len(predictions)}")
        print(f"📁 Submission saved to: {config['submission_path']}")
        
        print("\n📝 Sample predictions:")
        display_cols = ['id', 'answer', 'answer_value', 'ref_id']
        available_cols = [col for col in display_cols if col in submission_df.columns]
        print(submission_df[available_cols].head(3).to_string())
        
        if config['use_sample_mode']:
            print(f"\n🔔 Note: This was run in SAMPLE MODE")
            print("   Set config['use_sample_mode'] = False for full processing")
        
        print("\n🎉 Ready to submit to Kaggle!")
        print("="*60)
        
        return submission_df
        
    except Exception as e:
        print(f"❌ Prediction generation failed: {e}")
        return None

In [21]:
if __name__ == "__main__":
    submission_df = main()
    
    if submission_df is not None:
        print("✅ Kaggle process completed successfully!")
        print("Your submission.csv file is ready in /kaggle/working/")
    else:
        print("❌ Process failed!")

 WATTBOT RAG SYSTEM - COMPLETE WORKING VERSION

🏗️ Environment Check:
  Working directory: /workspaces/WattBot
  Expected input path: WattBot
  Output path: ./submission.csv

📂 Found directory: ML-Marathon-WattBot
   Files: []

📂 Found directory: data
   Files: ['metadata.csv', 'test_Q.csv', 'train_QA.csv']
✅ Using files from: data

📁 Loading data files with encoding detection...
Trying to read data/metadata.csv with utf-8 encoding...
Trying to read data/metadata.csv with latin-1 encoding...
✅ Successfully read with latin-1 encoding
Trying to read data/test_Q.csv with utf-8 encoding...
✅ Successfully read with utf-8 encoding
Trying to read data/train_QA.csv with utf-8 encoding...
✅ Successfully read with utf-8 encoding
  📄 Documents: 32
  🎯 Test questions: 282
  ❓ Training questions: 41

📋 Sample metadata columns: ['id', 'type', 'title', 'year', 'citation', 'url']
📋 Sample test question columns: ['id', 'question', 'answer', 'answer_value', 'answer_unit', 'ref_id', 'ref_url', 'supportin

Device set to use cpu


✅ Loaded QA model: distilbert-base-cased-distilled-squad

📚 Processing documents...
Loading cached chunks...
✅ Loaded 1232 cached chunks

🧮 Creating embeddings...
Loading cached embeddings...
✅ Loaded cached embeddings: (1232, 384)
🔍 Building search index...
✅ Index built: 1232 vectors

🎯 Generating predictions for 282 questions...


Answering:   0%|          | 0/282 [00:00<?, ?it/s]


💾 Creating submission file...
🧹 Comprehensive null value cleaning...
Null values after comprehensive cleaning: 0

🔍 Final validation check:
  ✅ Column id looks clean
  ✅ Column question looks clean
  ✅ Column answer looks clean
  ✅ Column answer_value looks clean
  ✅ Column answer_unit looks clean
  ✅ Column ref_id looks clean
  ✅ Column ref_url looks clean
  ✅ Column supporting_materials looks clean
  ✅ Column explanation looks clean

 KAGGLE PROCESSING COMPLETE!
⏱️  Total time: 0:01:16.964252
📊 Documents processed: 31
📄 Text chunks created: 1232
✅ Questions answered: 282
📁 Submission saved to: ./submission.csv

📝 Sample predictions:
     id                                                             answer answer_value                                  ref_id
0  q001  Unable to answer with confidence based on the provided documents.     is_blank                                is_blank
1  q002                                                               five         five  dodge2022, 